# Battery Model Dispatch Plot

This notebook loads either real electricity prices or generated price-model data and plots RL battery dispatch against the optimization baseline together with the electricity price.

In [6]:
import os
import sys
from pathlib import Path

os.environ["MPLBACKEND"] = "Agg"  # must be before pyplot import

import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in (start, *start.parents):
        if (candidate / "src").exists() and (candidate / "Battery_model").exists():
            return candidate
    raise FileNotFoundError("Could not find the THESIS repository root.")


ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

PRICE_FILE = (
    ROOT
    / "Battery_model"
    / "Optimization_Pyomo"
    / "input_data"
    / "electricity_price.txt"
)
# Choose one or both algorithms, e.g. ("SAC",) or ("SAC", "TD3").
RL_ALGORITHMS = ("TD3",)
RL_MODEL_PATHS = {
    "SAC": ROOT / "src" / "battery_model" / "battery_sac_terminal",
    "TD3": ROOT / "src" / "battery_model" / "battery_td3_terminal",
}
OUTPUT_FIGURE = ROOT / "notebooks" / "battery_model_dispatch_plot.png"

# Choose price source: "real" or "model".
PRICE_SOURCE = "real"
PRICE_MODEL_HOURS = 8784
PRICE_MODEL_SEED = None  # Set an integer for a reproducible stochastic path.
PRICE_MODEL_INITIAL_PRICE = None
PRICE_MODEL_START_HOUR = 0.0
PRICE_MODEL_DT_HOURS = 1.0
PRICE_MODEL_PRICE_VOLATILITY_SCALE = 50.0
PRICE_MODEL_WIND_VOLATILITY_SCALE = 1.0

# Use one day for the dispatch comparison plot.
WINDOW_START = 0
WINDOW_HOURS = 24

plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)

ROOT


PosixPath('/home/ashish/Documents/THESIS')

In [7]:
price_source = PRICE_SOURCE.lower()

if price_source == "real":
    price_df = pd.read_csv(
        PRICE_FILE,
        sep=r"\s+",
        header=None,
        names=["timestamp_s", "price_eur_per_mwh"],
    )
    prices_full = price_df["price_eur_per_mwh"].astype(float).to_numpy()
    GLOBAL_PRICE_SCALE = np.percentile(np.abs(prices_full), 95) + 1e-6
    PRICE_SOURCE_LABEL = f"real prices from {PRICE_FILE.relative_to(ROOT)}"
elif price_source == "model":
    from src.battery_model.price_model import load_price_model_data

    prices_full, GLOBAL_PRICE_SCALE = load_price_model_data(
        num_hours=PRICE_MODEL_HOURS,
        initial_price=PRICE_MODEL_INITIAL_PRICE,
        start_hour=PRICE_MODEL_START_HOUR,
        dt_hours=PRICE_MODEL_DT_HOURS,
        seed=PRICE_MODEL_SEED,
    )
    price_df = pd.DataFrame(
        {
            "timestamp_s": np.arange(len(prices_full)) * PRICE_MODEL_DT_HOURS * 3600.0,
            "price_eur_per_mwh": prices_full,
        }
    )
    PRICE_SOURCE_LABEL = (
        f"coupled wind-price model, seed={PRICE_MODEL_SEED}, "
        f"price_vol_scale={PRICE_MODEL_PRICE_VOLATILITY_SCALE}"
    )
else:
    raise ValueError('PRICE_SOURCE must be either "real" or "model".')

window_end = min(WINDOW_START + WINDOW_HOURS, len(prices_full))
prices_window = prices_full[WINDOW_START:window_end]

print(f"Loaded {len(prices_full)} hourly prices from {PRICE_SOURCE_LABEL}")
print(f"Using hours {WINDOW_START} to {window_end - 1} ({len(prices_window)} hours)")
print(f"Global price scale: {GLOBAL_PRICE_SCALE:.4f}")

price_df.iloc[WINDOW_START:window_end]


Loaded 8784 hourly prices from real prices from Battery_model/Optimization_Pyomo/input_data/electricity_price.txt
Using hours 0 to 23 (24 hours)
Global price scale: 65.5685


,timestamp_s,price_eur_per_mwh
0,0,25.12
1,3600,23.64
2,7200,19.03
3,10800,16.00
4,14400,16.00
5,18000,9.49
6,21600,-0.01
7,25200,-0.08
8,28800,-0.03
9,32400,12.09


In [8]:
def build_dispatch_frame(
    method, prices, power_kw, soc, power_actual_kw=None, control=None, rewards=None
):
    prices = np.asarray(prices, dtype=float)
    power_kw = np.asarray(power_kw, dtype=float)
    soc = np.asarray(soc, dtype=float)

    data = {
        "hour": np.arange(len(prices)),
        "price_eur_per_mwh": prices,
        "power_kw": power_kw,
        "charge_kw": np.clip(power_kw, 0.0, None),
        "discharge_kw": np.clip(-power_kw, 0.0, None),
        "soc": soc,
        "method": method,
    }
    if power_actual_kw is not None:
        data["power_actual_kw"] = np.asarray(power_actual_kw, dtype=float)
    if control is not None:
        data["control_u"] = np.asarray(control, dtype=float)
    if rewards is not None:
        data["reward"] = np.asarray(rewards, dtype=float)

    return pd.DataFrame(data)


def run_rl_dispatch(algorithm, prices, price_scale):
    from stable_baselines3 import SAC, TD3

    from src.battery_model.evaluate import evaluate_model

    algorithm = algorithm.upper()
    model_classes = {"SAC": SAC, "TD3": TD3}
    if algorithm not in model_classes:
        raise ValueError(f"Unsupported algorithm: {algorithm}. Choose from {tuple(model_classes)}.")

    model_path = RL_MODEL_PATHS[algorithm]
    if not model_path.with_suffix(".zip").exists():
        raise FileNotFoundError(
            f"Missing saved {algorithm} model: {model_path.with_suffix('.zip')}"
        )

    model = model_classes[algorithm].load(str(model_path))
    p_cmds, power_kw, power_actual_kw, soc, rewards, prices_used = evaluate_model(
        model,
        prices,
        price_scale,
    )
    return build_dispatch_frame(
        f"RL {algorithm}",
        prices_used,
        power_kw,
        soc,
        power_actual_kw=power_actual_kw,
        control=p_cmds / 20.0,
        rewards=rewards,
    )


def run_pyomo_dispatch(prices):
    from Battery_model.Optimization_Pyomo.pyomo_battery_model_v2 import (
        get_battery_model,
    )

    result = get_battery_model(prices, verbose=False)
    if result.get("status") != "success":
        raise RuntimeError(result.get("error", "Pyomo optimization failed."))

    return build_dispatch_frame(
        "optimization baseline",
        prices,
        result["P"],
        result["SOC"],
        power_actual_kw=result.get("P_actual"),
        control=result.get("u"),
    )


def objective_value_eur(df):
    return float(np.sum((df["price_eur_per_mwh"] / 1000.0) * df["power_kw"]))


def summarize_dispatch(df):
    objective_eur = objective_value_eur(df)
    return pd.DataFrame(
        [
            {
                "method": df["method"].iloc[0],
                "window_hours": len(df),
                "objective_value_eur": objective_eur,
                "total_cost_eur": objective_eur,
                "charge_energy_kwh": float(df["charge_kw"].sum()),
                "discharge_energy_kwh": float(df["discharge_kw"].sum()),
                "min_soc": float(df["soc"].min()),
                "max_soc": float(df["soc"].max()),
            }
        ]
    )


In [9]:
rl_dispatches = {
    algorithm.upper(): run_rl_dispatch(algorithm, prices_window, GLOBAL_PRICE_SCALE)
    for algorithm in RL_ALGORITHMS
}
# Backwards-compatible name for code that expects one primary RL dispatch.
dispatch_df = rl_dispatches[RL_ALGORITHMS[0].upper()]
math_dispatch_df = run_pyomo_dispatch(prices_window)

for algorithm in rl_dispatches:
    print(f"Prepared RL {algorithm} dispatch for the plot.")
print("Prepared the optimization-baseline dispatch for the plot.")

summary_frames = [summarize_dispatch(df) for df in rl_dispatches.values()]
summary_frames.append(summarize_dispatch(math_dispatch_df))

display(pd.concat(summary_frames, ignore_index=True))
dispatch_df.head()


Prepared RL TD3 dispatch for the plot.
Prepared the optimization-baseline dispatch for the plot.


,method,window_hours,objective_value_eur,total_cost_eur,charge_energy_kwh,discharge_energy_kwh,min_soc,max_soc
0,RL TD3,24,-2.328208,-2.328208,190.754501,152.510564,0.175287,1.000000
1,optimization baseline,24,-2.732450,-2.732450,197.684568,160.190956,0.298287,0.996987


,hour,price_eur_per_mwh,power_kw,charge_kw,discharge_kw,soc,method,power_actual_kw,control_u,reward
0,0,25.120001,-18.000000,0.000000,18.000000,0.400000,RL TD3,-20.000000,-0.050000,0.452160
1,1,23.639999,-18.000000,0.000000,18.000000,0.300000,RL TD3,-20.000000,-0.050000,0.425520
2,2,19.030001,-17.921144,0.000000,17.921144,0.200438,RL TD3,-19.912382,-0.049781,0.341039
3,3,16.000000,-4.527261,0.000000,4.527261,0.175287,RL TD3,-5.030178,-0.012576,0.072436
4,4,16.000000,18.756728,18.756728,0.000000,0.259692,RL TD3,16.881055,0.046892,-0.300108


In [10]:
def plot_battery_dispatch(df, math_df, output_path=None, initial_soc=0.5):
    hours = df["hour"].to_numpy(dtype=int)
    math_hours = math_df["hour"].to_numpy(dtype=int)
    x_min = int(min(hours.min(), math_hours.min()))
    x_max = int(max(hours.max(), math_hours.max())) + 1
    x_ticks = np.arange(x_min, x_max + 1, 1)

    rl_power = df["power_kw"].to_numpy(dtype=float)
    math_power = math_df["power_kw"].to_numpy(dtype=float)
    rl_power_hours = np.r_[hours, x_max]
    math_power_hours = np.r_[math_hours, x_max]
    rl_power_step = np.r_[rl_power, rl_power[-1]]
    math_power_step = np.r_[math_power, math_power[-1]]
    rl_soc_hours = np.r_[x_min, hours + 1]
    math_soc_hours = np.r_[x_min, math_hours + 1]
    rl_soc = np.r_[initial_soc, df["soc"].to_numpy(dtype=float)] * 100.0
    math_soc = np.r_[initial_soc, math_df["soc"].to_numpy(dtype=float)] * 100.0
    price_hours = np.r_[hours, x_max]
    price_values = np.r_[
        df["price_eur_per_mwh"].to_numpy(dtype=float), df["price_eur_per_mwh"].iloc[-1]
    ]

    max_abs_power = max(
        float(np.max(np.abs(rl_power))),
        float(np.max(np.abs(math_power))),
        1.0,
    )

    fig, (ax_power, ax_soc) = plt.subplots(
        2,
        1,
        figsize=(14, 6.5),
        sharex=True,
        gridspec_kw={"height_ratios": [3.0, 1.0], "hspace": 0.08},
    )

    rl_area = ax_power.fill_between(
        rl_power_hours,
        0.0,
        rl_power_step,
        step="post",
        color="#78a984",
        edgecolor="#0b5d1e",
        linewidth=1.0,
        alpha=0.72,
        label="RL charge/discharge",
    )
    ax_power.step(
        rl_power_hours, rl_power_step, where="post", color="#0b5d1e", linewidth=1.2
    )
    math_line = ax_power.step(
        math_power_hours,
        math_power_step,
        where="post",
        color="blue",
        linewidth=2.2,
        linestyle="--",
        label="Math charge/discharge",
    )[0]
    ax_power.axhline(0.0, color="#222222", linewidth=1.0)
    ax_power.set_ylim(-max_abs_power * 1.25, max_abs_power * 1.25)
    ax_power.set_xlim(x_min, x_max)
    ax_power.set_ylabel("Power [kW]", fontweight="bold")
    ax_power.grid(True, alpha=0.28)
    ax_power.tick_params(labelbottom=False)

    ax_price = ax_power.twinx()
    price_line = ax_price.plot(
        price_hours,
        price_values,
        color="red",
        linewidth=2.2,
        label="Electricity price",
    )[0]
    ax_price.set_ylabel("Electricity price [EUR/MWh]", fontweight="bold")

    rl_soc_line = ax_soc.plot(
        rl_soc_hours,
        rl_soc,
        color="#222222",
        linewidth=2.1,
        label="RL SOC",
    )[0]
    ax_soc.fill_between(rl_soc_hours, 0, rl_soc, color="#222222", alpha=0.08)
    math_soc_line = ax_soc.plot(
        math_soc_hours,
        math_soc,
        color="#5f5f5f",
        linewidth=2.1,
        linestyle="--",
        label="Math SOC",
    )[0]
    ax_soc.set_ylim(0, 100)
    ax_soc.set_xlim(x_min, x_max)
    ax_soc.set_xticks(x_ticks)
    ax_soc.set_xlabel("Time [h]")
    ax_soc.set_ylabel("SOC [%]", fontweight="bold")
    ax_soc.grid(True, alpha=0.28)

    rl_method = df["method"].iloc[0]
    title = f"Battery dispatch: {rl_method} vs. the optimization baseline ({PRICE_SOURCE}) | hours {WINDOW_START}-{window_end - 1}"
    objective_text = (
        f"{rl_method} objective: {objective_value_eur(df):.2f} EUR"
        f"  |  Baseline objective: {objective_value_eur(math_df):.2f} EUR"
    )
    fig.suptitle(title, fontsize=16, y=0.985)
    fig.text(0.5, 0.925, objective_text, ha="center", va="center", fontsize=11)
    fig.legend(
        [rl_area, math_line, price_line, rl_soc_line, math_soc_line],
        [
            f"{rl_method} charge/discharge",
            "Math charge/discharge",
            "Electricity price",
            f"{rl_method} SOC",
            "Math SOC",
        ],
        loc="upper center",
        bbox_to_anchor=(0.5, 0.875),
        ncol=5,
        frameon=True,
        fontsize=10,
    )
    fig.subplots_adjust(top=0.74)

    if output_path is not None:
        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(output_path, dpi=300, bbox_inches="tight")
        print(f"Saved figure to {output_path.relative_to(ROOT)}")

    return fig


for algorithm, algorithm_dispatch_df in rl_dispatches.items():
    output_path = (
        OUTPUT_FIGURE
        if algorithm == "SAC"
        else OUTPUT_FIGURE.with_name(
            f"{OUTPUT_FIGURE.stem}_{algorithm.lower()}{OUTPUT_FIGURE.suffix}"
        )
    )
    plot_battery_dispatch(algorithm_dispatch_df, math_dispatch_df, output_path)

Saved figure to notebooks/battery_model_dispatch_plot_td3.png


Set `PRICE_SOURCE = "real"` to use the input price file or `PRICE_SOURCE = "model"` to use the stochastic price model. Set `RL_ALGORITHMS = ("SAC",)`, `("TD3",)`, or `("SAC", "TD3")` to select trained policies. When both are selected, the notebook creates one plot per algorithm against the mathematical optimizer; the TD3 plot is saved as `battery_model_dispatch_plot_td3.png`. Pyomo requires a working IPOPT installation.